# STEP 7. 민감도 분석과 강건한 공원 묶음 (robust park groups)

**목적**: 하나의 임계값에 기대지 않고, 파라미터 격자 전체에서 **꾸준히 유지되는** 공원 묶음과 고립 공원을 찾는다.

**정의별 실질 파라미터 격자** (주 변형 C, 기본 attachment `d20_K8`, 면적 상한 100):
- N1abs: θ × R × {best, robust} = 24개 설정
- N1rel: q = 2개 설정
- D: d ∈ {200, 400, 800} m = 3개 설정 (근접성 기준선)
- N2: stroke 각도 × 간격 한계 = 9개 설정
- N3: θc × R × τ × κ = 16개 설정

**공원별 지표**
$$\text{IsolationPersistence}_i = \frac{\#\{\text{설정 중 } i \text{가 isolated}\}}{\#\text{설정}}$$
같은 방식으로 `GroupPersistence`(크기 3 이상 연결요소 소속)와 `CorePersistence`(2-core 소속)를 구한다.

**강건한 묶음**: 공원 쌍의 동반 소속 비율 $M_{ij}$ = 두 공원이 같은 **Louvain 커뮤니티**(크기 ≥ 2)에 속한 설정의 비율. 연결요소를 쓰면 관대한 설정에서 생기는 거대 연결요소(퍼콜레이션) 때문에 묶음이 하나로 뭉치므로, STEP 6의 커뮤니티를 쓴다. $1 - M$에 평균 연결(average linkage) 계층 군집을 적용하고 0.5(주)·0.3(엄격, 평균 동반소속 ≥ 0.7)에서 자른다. 평균 연결을 쓰면 단일 연결의 사슬 효과 없이 "대부분의 설정에서 함께 묶이는" 집합만 남는다.

**방법론적 민감도**: 네트워크 변형 C↔S, 출발점 K 8↔4, attachment 버퍼 d, 면적 상한 50/100/200, 접근 제한 공원 제외, 가로 매몰형 제외, E0 합집합.

**출력**: `persistence.parquet`, `robust_groups_{def}.parquet`, `sensitivity_*.csv`

In [1]:
# ===== 공통 설정 (모든 STEP 노트북에서 동일) =====
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 분석 지역: "Brooklyn"(파일럿) 또는 "NYC"(전체). 환경변수 PARK_STUDY_AREA로도 지정 가능.
STUDY_AREA = os.environ.get("PARK_STUDY_AREA", "Brooklyn")

ROOT = Path.cwd()
DATA = ROOT / "Data"
PARKS_PATH = DATA / "NYCPark" / "Parks_Properties_20260923.geojson"
NET_DIR = DATA / "derived" / "network"        # STEP1 결과 (지역 무관, NYC+버퍼 전체)
AREA_DIR = DATA / "derived" / STUDY_AREA      # STEP2 이후 결과
OUT_DIR = ROOT / "outputs" / STUDY_AREA
for _d in (NET_DIR, AREA_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

CRS = 32618            # UTM 18N (m)
R_MAX = 3200           # 탐색 한계 (m) = 보로 네트워크 버퍼 폭
SPEED = 1.0            # speed_m_s=1 -> agg_seconds == 경로 길이(m)

BOROUGHS = {"M": "Manhattan", "X": "Bronx", "B": "Brooklyn", "Q": "Queens", "R": "Staten Island"}
AREA_BOROUGHS = ["B"] if STUDY_AREA == "Brooklyn" else list(BOROUGHS)

# 네트워크 변형: C = 가로 중심선 + 보행전용 경로 (주 분석), S = 보도 포함 전체 보행망 (민감도)
VARIANTS = ["C", "S"]
PRIMARY = "C"
SIDEWALK_FOOTWAYS = {"sidewalk", "crossing", "traffic_island"}

# 공원 필터 (사용자 결정): Flagship + 비공원성 필지 제외, 면적 상한 100 acres (계산은 200 상위집합)
EXCLUDE_TYPES = ["Flagship Park", "Operations", "Lot", "Undeveloped",
                 "Buildings/Institutions", "Cemetery", "Managed Sites"]
AREA_CAP = 100
AREA_CAP_SUPERSET = 200
AREA_CAP_GRID = [50, 100, 200]
RESTRICTED_TYPES = ["Garden", "Jointly Operated Playground"]            # 접근 제한 민감도
STREET_EMBEDDED_TYPES = ["Triangle/Plaza", "Strip", "Mall", "Parkway"]  # 가로 매몰형 민감도

# 공원-네트워크 attachment
ATTACH_BUFFER = {"C": 20.0, "S": 10.0}   # 경계 버퍼 (m): C는 중심선이 도로 폭 절반만큼 떨어져 있음
ATTACH_BUFFER_GRID = {"C": [15.0, 20.0, 30.0], "S": [5.0, 10.0, 20.0]}
INSIDE_SHARE = 0.9       # 공원 내부에 90% 이상 들어간 세그먼트는 attachment에서 제외 (통행은 허용)
K_SOURCES = 8            # 공원당 출발 attachment 최대 개수 (farthest-point sampling)
SMALL_COMPONENT = 200    # 이보다 작은 연결요소에만 붙은 공원은 'unresolved'

# 최종 묶음 공통 절차: 설정별 Louvain(γ) → 보로 단위 퍼콜레이션 필터 → 합의 ≥ 0.5 → 도보 10분 지름 분할
MAIN_GAMMA = 2
COMMERCIAL_BUFFER = 100   # 연구 개요: 군집 구성 공원 주변 100 m = 상업 영역

# 묶음 크기 제약: 묶음 안 모든 공원 쌍이 서로 도보 10분 이내 (보행망 최단거리, 1.33 m/s × 600 s ≈ 800 m)
WALK_LIMIT = 800
WALK_LIMIT_GRID = [400, 800, 1200]   # 5 / 10 / 15분 민감도

# N1 angular few-turn
N1_THETAS = [15, 45, 90, 180]
N1_RADII = [800, 1600, 3200]
N1_REL_Q = [0.1, 0.2]
DIST_BANDS = [0, 200, 400, 800, 1600, 3200]

# D walking distance (근접성 기준선): 보행망 최단거리 ≤ d 이면 연결
D_DISTS = [200, 400, 800]

# N2 street-axis continuity: 같은 연속 가로(stroke) 위, 두 공원 구간 사이 간격 ≤ lim
N2_ALONG = [400, 800, 1600]
# (큰길 속성 계산용) 각도 choice/NACH 반경과 상위 비율
N2_RADII = [800, 1600, 3200]
N2_TOP_P = [0.02, 0.05, 0.10]
STROKE_ANGLE = 30
STROKE_ANGLE_GRID = [20, 30, 45]
N2_MAX_ALONG = [3200, None]

# N3 excess few-turn field overlap
N3_THETAS = [45, 90]
N3_RADII = [800, 1600]
N3_TAU = [0.1, 0.2]
N3_KAPPA = [1.0, 1.25]

print(f"STUDY_AREA={STUDY_AREA}  boroughs={AREA_BOROUGHS}  AREA_DIR={AREA_DIR.relative_to(ROOT)}")

STUDY_AREA=Brooklyn  boroughs=['B']  AREA_DIR=Data/derived/Brooklyn


In [2]:
import networkx as nx
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
from sklearn.metrics import adjusted_rand_score

GR_DIR = AREA_DIR / "graphs"
parks = gpd.read_parquet(NET_DIR / "parks.parquet")
cls = pd.read_parquet(GR_DIR / "classification.parquet")
settings = pd.read_csv(GR_DIR / "settings.csv")["setting"]
edges = pd.concat([pd.read_parquet(GR_DIR / f"edges_{d}.parquet") for d in ["N1", "N2", "N3"]], ignore_index=True)
EDGE_GROUPS = dict(tuple(edges.groupby("setting")))
EMPTY = edges.iloc[:0]
DEF_CFG = {v: f"d{int(ATTACH_BUFFER[v])}_K{K_SOURCES}" for v in VARIANTS}
DEFINITIONS = ["D", "N1abs", "N1rel", "N2", "N3"]


def family(definition, v=PRIMARY, cfg=None):
    """정의별 실질 파라미터 격자의 설정 목록."""
    s = settings[settings.str.startswith(f"{definition}|{v}|")]
    if definition.startswith("N1") or definition == "D":
        s = s[s.str.split("|").str[2] == (cfg or DEF_CFG[v])]
    return s.tolist()


for d in DEFINITIONS:
    print(d, len(family(d)), "settings")

D 3 settings
N1abs 24 settings
N1rel 2 settings
N2 9 settings
N3 16 settings


## 7-1. 공원별 persistence (주 변형, 상한 100)

In [3]:
def persistence(cls_sub, sets):
    c = cls_sub[cls_sub.setting.isin(sets)]
    g = c.groupby("park_id")
    return pd.DataFrame({
        "iso": g["role"].apply(lambda r: (r == "isolated").mean()),
        "unresolved": g["role"].apply(lambda r: (r == "unresolved").mean()),
        "grouped": g["comp_size"].apply(lambda s: (s >= 3).mean()),
        "core": g["role"].apply(lambda r: (r == "core").mean()),
        "bridge": g["articulation"].mean(),
        "degree_mean": g["degree"].mean(),
    })


main_cls = cls[(cls.cap == AREA_CAP) & (cls.variant == PRIMARY)]
pers = {d: persistence(main_cls, family(d)) for d in DEFINITIONS}
P = pd.concat(pers, axis=1)
P.columns = [f"{a}_{b}" for a, b in P.columns]
P.to_parquet(AREA_DIR / "persistence.parquet")
summary = pd.DataFrame({d: {
    "parks": len(pers[d]),
    "robust isolated (iso ≥ 0.8)": int((pers[d].iso >= 0.8).sum()),
    "mostly isolated (iso ≥ 0.5)": int((pers[d].iso >= 0.5).sum()),
    "robust grouped (grouped ≥ 0.8)": int((pers[d].grouped >= 0.8).sum()),
    "robust core (core ≥ 0.5)": int((pers[d].core >= 0.5).sum()),
    "unresolved (≥ 0.5)": int((pers[d].unresolved >= 0.5).sum()),
} for d in DEFINITIONS})
summary

,D,N1abs,N1rel,N2,N3
parks,621,621,621,621,621
robust isolated (iso ≥ 0.8),11,14,31,93,481
mostly isolated (iso ≥ 0.5),71,109,75,147,573
robust grouped (grouped ≥ 0.8),330,377,540,319,0
robust core (core ≥ 0.5),420,403,561,285,8
unresolved (≥ 0.5),2,2,2,2,2


## 7-2. 동반 소속 행렬과 강건한 공원 묶음

In [4]:
def comembership(cls_sub, sets, pids):
    pos = {p: k for k, p in enumerate(pids)}
    M = np.zeros((len(pids), len(pids)), dtype=np.float32)
    n = 0
    sub = cls_sub[cls_sub.setting.isin(sets) & cls_sub.park_id.isin(pos)]
    for s, c in sub.groupby("setting"):
        lab = np.full(len(pids), -1)
        idx = c.park_id.map(pos).values
        lab[idx] = c.community.values
        same = (lab[:, None] == lab[None, :]) & (lab[:, None] >= 0)
        M += same
        n += 1
    M /= max(n, 1)
    np.fill_diagonal(M, 1.0)
    return M


def consensus_groups(M, pids, cut):
    D = 1.0 - M
    np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), method="average")
    lab = fcluster(Z, t=cut, criterion="distance")
    s = pd.Series(lab, index=pids)
    size = s.map(s.value_counts())
    return s.where(size >= 2, -1), size.where(size >= 2, 1)


PIDS = sorted(main_cls.park_id.unique())
GROUPS = {}
grp_summary = []
for d in DEFINITIONS:
    M = comembership(main_cls, family(d), PIDS)
    np.save(AREA_DIR / f"comembership_{d}.npy", M)
    out = pd.DataFrame(index=pd.Index(PIDS, name="park_id"))
    for cut, tag in [(0.5, "g50"), (0.3, "g70")]:
        lab, size = consensus_groups(M, PIDS, cut)
        out[f"group_{tag}"], out[f"size_{tag}"] = lab, size
    # 묶음 내부 평균 동반소속
    pos = {p: k for k, p in enumerate(PIDS)}
    coh = {}
    for gid, members in out[out.group_g50 >= 0].groupby("group_g50").groups.items():
        ix = [pos[m] for m in members]
        sub = M[np.ix_(ix, ix)]
        coh[gid] = (sub.sum() - len(ix)) / (len(ix) * (len(ix) - 1))
    out["group_cohesion"] = out.group_g50.map(coh)
    out = out.join(pers[d])
    out.to_parquet(AREA_DIR / f"robust_groups_{d}.parquet")
    GROUPS[d] = out
    sz = out[out.group_g50 >= 0].groupby("group_g50").size()
    grp_summary.append({"definition": d, "groups(g50)": len(sz), "parks in groups": int(sz.sum()),
                        "median size": sz.median() if len(sz) else 0, "max size": sz.max() if len(sz) else 0,
                        "groups(g70)": out[out.group_g70 >= 0].group_g70.nunique(),
                        "parks in g70": int((out.group_g70 >= 0).sum())})
pd.DataFrame(grp_summary)

,definition,groups(g50),parks in groups,median size,max size,groups(g70),parks in g70
0,D,70,544,4.0,45,89,397
1,N1abs,58,514,3.0,53,68,444
2,N1rel,6,588,105.5,132,21,540
3,N2,62,467,3.0,45,71,370
4,N3,22,56,2.0,8,14,29


## 7-3. 방법론적 민감도
기준(주 변형 C, 기본 cfg, 상한 100)과 대안 설정을 비교한다:
- **ARI**: 강건 묶음(g50) 라벨의 일치도(묶음에 들지 않은 공원은 각자 단일 군집)
- **iso r**: 공원별 IsolationPersistence의 상관
- **Δiso**: 평균 IsolationPersistence 차이(대안 − 기준)

In [5]:
def labels_with_singletons(s):
    s = s.copy()
    m = s < 0
    s[m] = -np.arange(1, m.sum() + 1)
    return s


def compare(d, alt_cls, alt_sets, label, pids=None):
    base = GROUPS[d]
    pids = pids or PIDS
    pids = [p for p in pids if p in set(alt_cls.park_id)]
    M = comembership(alt_cls, alt_sets, pids)
    lab, _ = consensus_groups(M, pids, 0.5)
    pa = persistence(alt_cls, alt_sets).reindex(pids)
    b = base.reindex(pids)
    return {"definition": d, "alternative": label, "n_settings": len(alt_sets),
            "ARI_groups": adjusted_rand_score(labels_with_singletons(b.group_g50), labels_with_singletons(lab)),
            "iso_r": np.corrcoef(b.iso, pa.iso)[0, 1] if pa.iso.std() > 0 and b.iso.std() > 0 else np.nan,
            "d_iso": pa.iso.mean() - b.iso.mean()}


sens = []
for d in DEFINITIONS:
    # 네트워크 변형 S
    other = [v for v in VARIANTS if v != PRIMARY][0]
    alt = cls[(cls.cap == AREA_CAP) & (cls.variant == other)]
    if family(d, v=other):   # N2는 주 변형에서만 계산
        sens.append(compare(d, alt, family(d, v=other), f"variant {other}"))
    if d.startswith("N1") or d == "D":
        for cfg in sorted(set(cls[cls.definition == d].setting.str.split("|").str[2]) - {DEF_CFG[PRIMARY]}):
            sets = family(d, cfg=cfg)
            if sets:
                sens.append(compare(d, main_cls, sets, f"attachment {cfg}"))
    for cap in AREA_CAP_GRID:
        if cap != AREA_CAP:
            alt = cls[(cls.cap == cap) & (cls.variant == PRIMARY)]
            common = sorted(set(alt.park_id) & set(PIDS))
            sens.append(compare(d, alt, family(d), f"area cap {cap}", pids=common))
sens = pd.DataFrame(sens)
sens.to_csv(AREA_DIR / "sensitivity_methodological.csv", index=False)
sens.round(3)

,definition,alternative,n_settings,ARI_groups,iso_r,d_iso
0,D,variant S,3,0.840,0.946,0.009
1,D,area cap 50,3,0.979,0.973,0.004
2,D,area cap 200,3,0.988,0.956,-0.009
3,N1abs,variant S,24,0.391,0.580,0.205
4,N1abs,attachment d15_K8,24,0.825,0.850,0.050
5,N1abs,attachment d20_K4,24,0.959,0.988,0.009
6,N1abs,attachment d30_K8,24,0.855,0.902,-0.027
7,N1abs,area cap 50,24,0.919,0.987,0.006
8,N1abs,area cap 200,24,0.914,0.977,-0.007
9,N1rel,variant S,2,0.793,0.384,-0.007


## 7-4. 노드 집합 민감도: 접근 제한 공원, 가로 매몰형 공원 제외, E0 합집합
노드를 빼거나 edge를 더하면 그래프가 바뀌므로 다시 분류한다(`classify_graph`는 STEP 6과 동일).

In [6]:
def classify_graph(nodes, e, unresolved):
    G = nx.Graph()
    G.add_nodes_from(nodes)
    G.add_edges_from(zip(e.i.values, e.j.values))
    core = nx.core_number(G)
    art = set(nx.articulation_points(G))
    rows = {}
    for cid, comp in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)):
        size = len(comp)
        has_core = any(core[n] >= 2 for n in comp)
        for n in comp:
            role = ("isolated" if size == 1 else "dyad" if size == 2 else "core" if core[n] >= 2
                    else "periphery" if has_core else "chain")
            rows[n] = (cid if size > 1 else -1, size, G.degree(n), core[n], n in art, role)
    out = pd.DataFrame.from_dict(rows, orient="index",
                                 columns=["comp", "comp_size", "degree", "core_number", "articulation", "role"])
    out.loc[out.index.isin(unresolved) & (out.degree == 0), "role"] = "unresolved"
    out["community"], out["community_size"] = -1, 1
    if G.number_of_edges():
        for k, cm in enumerate(sorted(nx.community.louvain_communities(G, seed=0), key=len, reverse=True)):
            if len(cm) > 1:
                out.loc[list(cm), "community"] = k
                out.loc[list(cm), "community_size"] = len(cm)
    return out.rename_axis("park_id").reset_index()


e0 = pd.read_parquet(NET_DIR / "pairs_E0.parquet")
e0_edges = e0[(e0[f"shared_{PRIMARY}"] > 0) | e0.touch][["i", "j"]]
unres = set(parks.index[parks[f"unresolved_{PRIMARY}"]])
base_nodes = PIDS
scen = {
    "exclude restricted (Garden, JOP)": [p for p in base_nodes if not parks.at[p, "restricted"]],
    "exclude street-embedded": [p for p in base_nodes if not parks.at[p, "street_embedded"]],
}
node_sens = []
for d in DEFINITIONS:
    sets = family(d)
    for label, nodes in list(scen.items()) + [("union with E0", base_nodes)]:
        ns = set(nodes)
        rows = []
        for s in sets:
            e = EDGE_GROUPS.get(s, EMPTY)
            if label == "union with E0":
                e = pd.concat([e[["i", "j"]], e0_edges])
            e = e[e.i.isin(ns) & e.j.isin(ns)]
            c = classify_graph(nodes, e, unres)
            c["setting"] = s
            rows.append(c)
        alt = pd.concat(rows)
        node_sens.append(compare(d, alt, sets, label, pids=sorted(ns)))
node_sens = pd.DataFrame(node_sens)
node_sens.to_csv(AREA_DIR / "sensitivity_nodes.csv", index=False)
node_sens.round(3)

,definition,alternative,n_settings,ARI_groups,iso_r,d_iso
0,D,"exclude restricted (Garden, JOP)",3,0.000,0.047,0.823
1,D,exclude street-embedded,3,0.000,0.042,0.829
2,D,union with E0,3,0.132,0.399,0.503
3,N1abs,"exclude restricted (Garden, JOP)",24,0.799,0.845,0.060
4,N1abs,exclude street-embedded,24,0.703,0.910,0.033
5,N1abs,union with E0,24,0.857,0.829,-0.041
6,N1rel,"exclude restricted (Garden, JOP)",2,0.661,0.923,0.017
7,N1rel,exclude street-embedded,2,0.798,0.901,0.024
8,N1rel,union with E0,2,0.893,0.774,-0.033
9,N2,"exclude restricted (Garden, JOP)",9,0.750,0.841,0.087


## 7-5. 정의 간 비교: 같은 공원들이 묶이는가?
정의 쌍별로 강건 묶음 라벨 ARI와 IsolationPersistence 상관을 비교한다. 값이 낮을수록 두 정의가 **서로 다른 configurational 측면**을 잡는다는 뜻이다(연구 기여 포인트).

In [7]:
cross = []
for a, b in [(x, y) for i, x in enumerate(DEFINITIONS) for y in DEFINITIONS[i + 1:]]:
    ga, gb = GROUPS[a].reindex(PIDS), GROUPS[b].reindex(PIDS)
    cross.append({"A": a, "B": b,
                  "ARI_groups": adjusted_rand_score(labels_with_singletons(ga.group_g50), labels_with_singletons(gb.group_g50)),
                  "iso_r": np.corrcoef(ga.iso, gb.iso)[0, 1],
                  "both robust isolated": int(((ga.iso >= 0.8) & (gb.iso >= 0.8)).sum()),
                  "both robust grouped": int(((ga.grouped >= 0.8) & (gb.grouped >= 0.8)).sum())})
cross = pd.DataFrame(cross)
cross.to_csv(AREA_DIR / "cross_definition.csv", index=False)
cross.round(3)

,A,B,ARI_groups,iso_r,both robust isolated,both robust grouped
0,D,N1abs,0.462,0.380,5,259
1,D,N1rel,0.157,0.069,1,293
2,D,N2,0.516,0.495,7,254
3,D,N3,0.019,0.107,11,0
4,N1abs,N1rel,0.242,0.528,6,365
5,N1abs,N2,0.649,0.873,13,288
6,N1abs,N3,0.018,0.226,14,0
7,N1rel,N2,0.169,0.383,23,293
8,N1rel,N3,0.003,0.199,31,0
9,N2,N3,0.020,0.203,93,0
